# Health Risk Classification — Diabetes Prediction
**Project Name:** Health Risk Classification using SVM and Decision Tree
**Name:** Ali Siddiqui
**Roll No:** 24f-AI-095
**Section:** B
**Lab:** Lab 7 — Open Ended Lab
**Dataset:** Pima Indians Diabetes Database (`diabetes.csv`)

---
## Objective
Predict whether a patient is likely to have diabetes from basic health indicators (glucose, BMI, age, etc.).
The project uses **data preprocessing, visualization, two ML models (SVM and Decision Tree) and evaluation**, then compares the models and discusses **overfitting and the accuracy trade-off**.

## Steps
1. Load the dataset and describe it
2. Exploratory data analysis (EDA)
3. Preprocessing: handle missing values, outliers, normalize numeric data
4. Train SVM and Decision Tree (baseline, then tuned)
5. Evaluate: accuracy, precision, recall, F1, ROC-AUC, confusion matrix
6. Overfitting analysis
7. Model comparison and discussion

## 1. Import Libraries

In [ ]:
# Project: Health Risk Classification | Ali Siddiqui | 24f-AI-095 | Section B

# basic libraries for data and plots
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn tools: splitting, scaling, the two models and the metrics
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

# hide warnings so the output stays clean
import warnings
warnings.filterwarnings("ignore")

## 2. Dataset Description
The **Pima Indians Diabetes Database** contains medical records of Pima Indian women aged 21 or older.

| Feature | Description |
|---|---|
| Pregnancies | Number of times pregnant |
| Glucose | Plasma glucose concentration (2 hrs in an oral glucose tolerance test) |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Triceps skin fold thickness (mm) |
| Insulin | 2-hour serum insulin (mu U/ml) |
| BMI | Body mass index (kg/m²) |
| DiabetesPedigreeFunction | Score of diabetes family history |
| Age | Age in years |
| **Outcome** | **Target: 1 = diabetic, 0 = non-diabetic** |

In [ ]:
# load the data (on Colab, upload diabetes.csv first)
df = pd.read_csv("diabetes.csv")

# how many rows and columns?
print("Shape:", df.shape)
df.head()

In [ ]:
# data types and null values
df.info()

In [ ]:
# summary statistics (look at the min values for impossible zeros)
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
# is the data balanced? (diabetic vs non-diabetic)
print(df["Outcome"].value_counts())

sns.countplot(x="Outcome", data=df)
plt.title("Class Distribution (0 = Non-diabetic, 1 = Diabetic)")
plt.show()

In [ ]:
# histogram of every feature
df.hist(bins=20, figsize=(14, 10))
plt.show()

In [ ]:
# correlation heatmap: which features are related to each other and to Outcome?
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# boxplots to spot outliers
df.drop("Outcome", axis=1).plot(kind="box", subplots=True, layout=(2, 4), figsize=(16, 8))
plt.show()

## 4. Data Preprocessing
Steps (in this order):
1. **Split** the data into train and test sets first, so the test data does not influence preprocessing
2. **Missing values:** a 0 in Glucose, BloodPressure, SkinThickness, Insulin or BMI is impossible, so it really means "missing". Convert those zeros to NaN and fill them with the **median of the training set**
3. **Outliers:** cap extreme values using the IQR rule (we only have 768 rows, so we don't delete any)
4. **Normalization:** scale features with `StandardScaler` (needed for SVM)

> The provided CSV may already have the missing values filled in. The code works either way.

In [ ]:
# separate the inputs (X) from the target (y)
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# 80% train / 20% test; stratify keeps the same diabetic ratio in both
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

In [ ]:
# columns where 0 is not a real value
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Zeros in training data before cleaning:")
print((X_train[zero_cols] == 0).sum())

# change those zeros to NaN, then fill with the training median
for col in zero_cols:
    X_train[col] = X_train[col].replace(0, np.nan)
    X_test[col] = X_test[col].replace(0, np.nan)

    median = X_train[col].median()      # median comes from training data only
    X_train[col] = X_train[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

print("\nMissing values left:", X_train.isnull().sum().sum())

In [ ]:
# cap outliers using the IQR rule (limits are calculated on the training data)
for col in X_train.columns:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

X_train.describe()

In [ ]:
# scale the data: fit on train only, then use the same scaler on test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SVM will use the scaled data. The Decision Tree does not need scaling,
# so it will use the unscaled X_train / X_test (this also keeps the tree readable).
print("Scaled train mean (about 0):", X_train_scaled.mean().round(2))

## 5. Model Training

### 5.1 Baseline models
* **SVM** with the default RBF kernel
* **Decision Tree** with *no depth limit* (to show overfitting)

In [ ]:
# baseline SVM (probability=True so we can draw the ROC curve later)
svm_base = SVC(kernel="rbf", probability=True, random_state=42)
svm_base.fit(X_train_scaled, y_train)

# baseline Decision Tree with no limits on purpose
dt_base = DecisionTreeClassifier(random_state=42)
dt_base.fit(X_train, y_train)

# compare train accuracy with test accuracy
svm_train = svm_base.score(X_train_scaled, y_train)
svm_test = svm_base.score(X_test_scaled, y_test)
dt_train = dt_base.score(X_train, y_train)
dt_test = dt_base.score(X_test, y_test)

print("SVM baseline  -> train:", round(svm_train, 3), " test:", round(svm_test, 3))
print("Tree baseline -> train:", round(dt_train, 3), " test:", round(dt_test, 3))

The unpruned Decision Tree gets 100% on the training data but does worse on the test data. It memorized the training set, which is **overfitting**. Next we tune both models.

### 5.2 Tuning with cross-validation
We try different settings and pick the best one using **5-fold cross-validation on the training set only** (the test set stays untouched).

In [ ]:
# SVM: try different kernels and C values
best_score = 0
best_kernel = None
best_C = None

for kernel in ["linear", "rbf"]:
    for C in [0.1, 1, 10, 100]:
        model = SVC(kernel=kernel, C=C)
        score = cross_val_score(model, X_train_scaled, y_train, cv=5).mean()
        print("SVM kernel =", kernel, " C =", C, " CV accuracy =", round(score, 3))
        if score > best_score:
            best_score = score
            best_kernel = kernel
            best_C = C

print("\nBest SVM -> kernel:", best_kernel, " C:", best_C, " CV accuracy:", round(best_score, 3))

# train the final SVM with the best settings
svm_tuned = SVC(kernel=best_kernel, C=best_C, probability=True, random_state=42)
svm_tuned.fit(X_train_scaled, y_train)

In [ ]:
# Decision Tree: try different depths and minimum leaf sizes (this is called pruning)
best_score = 0
best_depth = None
best_leaf = None

for depth in [2, 3, 4, 5, 6, 8, None]:
    for leaf in [1, 5, 10]:
        model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=42)
        score = cross_val_score(model, X_train, y_train, cv=5).mean()
        if score > best_score:
            best_score = score
            best_depth = depth
            best_leaf = leaf

print("Best Tree -> max_depth:", best_depth, " min_samples_leaf:", best_leaf,
      " CV accuracy:", round(best_score, 3))

# train the final tree with the best settings
dt_tuned = DecisionTreeClassifier(max_depth=best_depth, min_samples_leaf=best_leaf, random_state=42)
dt_tuned.fit(X_train, y_train)

## 6. Model Evaluation
All four models are tested on the **test set**. In a medical problem, **recall** is very important: a false negative means a sick patient is told they are healthy.

In [ ]:
# function that calculates all metrics for one model
def evaluate(name, model, X_tr, X_te):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    train_acc = accuracy_score(y_train, model.predict(X_tr))
    test_acc = accuracy_score(y_test, y_pred)
    return [name, train_acc, test_acc,
            precision_score(y_test, y_pred), recall_score(y_test, y_pred),
            f1_score(y_test, y_pred), roc_auc_score(y_test, y_prob), train_acc - test_acc]

results = []
results.append(evaluate("SVM (baseline)", svm_base, X_train_scaled, X_test_scaled))
results.append(evaluate("SVM (tuned)", svm_tuned, X_train_scaled, X_test_scaled))
results.append(evaluate("Decision Tree (unpruned)", dt_base, X_train, X_test))
results.append(evaluate("Decision Tree (tuned)", dt_tuned, X_train, X_test))

results = pd.DataFrame(results, columns=["Model", "Train Acc", "Test Acc", "Precision",
                                         "Recall", "F1", "ROC-AUC", "Train-Test Gap"])
display(results.round(3))

In [ ]:
# predictions of the two tuned models on the test set
svm_pred = svm_tuned.predict(X_test_scaled)
dt_pred = dt_tuned.predict(X_test)

# confusion matrices side by side
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(y_test, svm_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-diabetic", "Diabetic"], yticklabels=["Non-diabetic", "Diabetic"])
plt.title("SVM (tuned)")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(y_test, dt_pred), annot=True, fmt="d", cmap="Greens",
            xticklabels=["Non-diabetic", "Diabetic"], yticklabels=["Non-diabetic", "Diabetic"])
plt.title("Decision Tree (tuned)")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# precision / recall / f1 for each class
print("SVM (tuned)")
print(classification_report(y_test, svm_pred, target_names=["Non-diabetic", "Diabetic"]))

print("Decision Tree (tuned)")
print(classification_report(y_test, dt_pred, target_names=["Non-diabetic", "Diabetic"]))

In [ ]:
# ROC curves (closer to the top-left corner = better)
svm_prob = svm_tuned.predict_proba(X_test_scaled)[:, 1]
dt_prob = dt_tuned.predict_proba(X_test)[:, 1]

fpr1, tpr1, _ = roc_curve(y_test, svm_prob)
fpr2, tpr2, _ = roc_curve(y_test, dt_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr1, tpr1, label="SVM (AUC = " + str(round(roc_auc_score(y_test, svm_prob), 3)) + ")")
plt.plot(fpr2, tpr2, label="Decision Tree (AUC = " + str(round(roc_auc_score(y_test, dt_prob), 3)) + ")")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()

In [ ]:
# the test set is small (154 rows), so also check cross-validation accuracy on all the data
# (SVM needs scaled data, so we scale everything with the training scaler just for this check)
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

svm_cv = cross_val_score(SVC(kernel=best_kernel, C=best_C), scaler.transform(X_all), y_all, cv=5)
dt_cv = cross_val_score(DecisionTreeClassifier(max_depth=best_depth, min_samples_leaf=best_leaf,
                                               random_state=42), X_all, y_all, cv=5)

print("SVM 5-fold CV accuracy:", round(svm_cv.mean(), 3), "+/-", round(svm_cv.std(), 3))
print("Tree 5-fold CV accuracy:", round(dt_cv.mean(), 3), "+/-", round(dt_cv.std(), 3))

## 7. Overfitting Analysis

### 7.1 Decision Tree: effect of tree depth

In [ ]:
# train trees with depth 1 to 15 and record train and test accuracy
depth_list = list(range(1, 16))
train_scores = []
test_scores = []

for d in depth_list:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(depth_list, train_scores, "o-", label="Train accuracy")
plt.plot(depth_list, test_scores, "s-", label="Test accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: Overfitting vs Depth")
plt.legend()
plt.show()

### 7.2 SVM: effect of C (RBF kernel)

In [ ]:
# small C = smooth boundary, large C = tries to fit every training point
C_list = [0.01, 0.1, 1, 10, 100, 1000]
train_scores = []
test_scores = []

for C in C_list:
    svm = SVC(kernel="rbf", C=C)
    svm.fit(X_train_scaled, y_train)
    train_scores.append(svm.score(X_train_scaled, y_train))
    test_scores.append(svm.score(X_test_scaled, y_test))

plt.figure(figsize=(8, 5))
plt.semilogx(C_list, train_scores, "o-", label="Train accuracy")
plt.semilogx(C_list, test_scores, "s-", label="Test accuracy")
plt.xlabel("C (log scale)")
plt.ylabel("Accuracy")
plt.title("SVM: Overfitting vs C")
plt.legend()
plt.show()

## 8. Data Quality Check: Possible Target Leakage in `Insulin`
The tree's feature importance (next section) shows `Insulin` dominating, which looks suspicious. In the provided CSV the missing insulin values seem to have been filled with a **different constant for each class**, so the filled value itself reveals the label (*target leakage*). That can make accuracy look better than it really is.

In [ ]:
# most common Insulin value for each class
print("Non-diabetic (0):")
print(df[df["Outcome"] == 0]["Insulin"].value_counts().head(1))
print("\nDiabetic (1):")
print(df[df["Outcome"] == 1]["Insulin"].value_counts().head(1))

In [ ]:
# retrain simple models with and without the suspicious columns to see the effect
def quick_test(drop_cols):
    xtr = X_train.drop(columns=drop_cols)
    xte = X_test.drop(columns=drop_cols)

    sc = StandardScaler()
    xtr_s = sc.fit_transform(xtr)
    xte_s = sc.transform(xte)

    svm = SVC(kernel="rbf", random_state=42).fit(xtr_s, y_train)
    tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42).fit(xtr, y_train)
    return round(svm.score(xte_s, y_test), 3), round(tree.score(xte, y_test), 3)

print("All features                     (SVM, Tree):", quick_test([]))
print("Without Insulin                  (SVM, Tree):", quick_test(["Insulin"]))
print("Without Insulin and SkinThickness(SVM, Tree):", quick_test(["Insulin", "SkinThickness"]))

## 9. Model Interpretation

In [ ]:
# which features does the tuned tree use the most?
importance = pd.Series(dt_tuned.feature_importances_, index=X.columns)
importance.sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Decision Tree Feature Importance")
plt.show()

In [ ]:
# draw the top of the tuned tree
plt.figure(figsize=(22, 10))
plot_tree(dt_tuned, feature_names=X.columns, class_names=["Non-diabetic", "Diabetic"],
          filled=True, max_depth=3, fontsize=9)
plt.show()

## 10. Model Comparison

In [ ]:
# compare the two tuned models side by side
compare = results[results["Model"].isin(["SVM (tuned)", "Decision Tree (tuned)"])]
compare = compare.set_index("Model")[["Test Acc", "Precision", "Recall", "F1", "ROC-AUC"]]
display(compare.round(3))

compare.plot(kind="bar", figsize=(9, 5), rot=0)
plt.title("Tuned SVM vs Tuned Decision Tree")
plt.ylim(0, 1)
plt.show()

## 11. Predicting Risk for a New Patient

In [ ]:
# a made-up patient to test both models
new_patient = pd.DataFrame([{
    "Pregnancies": 2, "Glucose": 150, "BloodPressure": 80, "SkinThickness": 30,
    "Insulin": 160, "BMI": 34.0, "DiabetesPedigreeFunction": 0.6, "Age": 45
}])

# SVM needs the scaled version, the tree uses the normal values
svm_risk = svm_tuned.predict_proba(scaler.transform(new_patient))[0][1]
tree_risk = dt_tuned.predict_proba(new_patient)[0][1]
average_risk = (svm_risk + tree_risk) / 2     # simple average of both models

print("SVM risk probability          :", round(svm_risk * 100, 1), "%")
print("Decision Tree risk probability:", round(tree_risk * 100, 1), "%")
print("Combined risk                 :", round(average_risk * 100, 1), "%")

if average_risk >= 0.5:
    print("Result: HIGH RISK")
else:
    print("Result: LOW RISK")

## 12. Results and Discussion

### Results (test set of 154 patients, `random_state=42`)
| Model | Train Acc | Test Acc | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| SVM (baseline) | 0.907 | 0.844 | 0.778 | 0.778 | 0.913 |
| SVM (tuned) | 0.907 | 0.844 | 0.778 | 0.778 | 0.913 |
| Decision Tree (unpruned) | 1.000 | 0.831 | 0.759 | 0.759 | 0.815 |
| Decision Tree (tuned) | 0.928 | 0.883 | 0.852 | 0.836 | 0.902 |

*(If your numbers are slightly different when you run it, update this table from your own output.)*

### Overfitting and the accuracy trade-off
* **Unpruned Decision Tree:** 100% accuracy on training data but only about 83% on test data (gap of about 17 points) and the lowest ROC-AUC (0.815). It memorized the training data, which is overfitting.
* **Tuned Decision Tree:** limiting the depth and setting a minimum leaf size reduced the gap to about 4.5 points, and test accuracy and recall went up (88.3% and 85.2%). We accept slightly lower training accuracy to get better results on new data.
* **SVM:** it generalizes fairly well even with default settings (gap about 6 points). Tuning picked the RBF kernel with C = 1, which is the same as the default, so the tuned SVM equals the baseline. The C plot shows that a very large C pushes training accuracy toward 100% while test accuracy does not improve, which is overfitting too.
* **Which model is better?** The tuned Decision Tree had higher accuracy and higher recall for diabetic patients (85% vs 78%), which matters because missing a sick patient is the costly mistake. The SVM had a slightly higher ROC-AUC (0.913 vs 0.902). The test set is small, so the accuracy difference is only about 6 patients, and the cross-validation scores (0.861 vs 0.848) are close. Overall the two models perform **similarly**. The tree is easier to explain; the SVM is smoother and usually less sensitive to small changes in the data.

### Important note about the data
Section 8 shows that the missing `Insulin` values in this CSV were filled with a different constant for each class, so the filled value leaks the label. This is why `Insulin` is by far the most important feature (about 70%) and why accuracy here is higher than normally reported for this dataset (about 75-78%). Without `Insulin`, test accuracy falls to about 75% (SVM) and 79% (tree). So the real-world performance would be lower than the numbers above, and these models are for learning only, not for clinical use.

### Conclusion
Both models predict diabetes risk reasonably well from basic health indicators. Cleaning the data (median imputation, outlier capping, scaling) and tuning with cross-validation were important, especially to control overfitting in the Decision Tree. Glucose and age are medically meaningful predictors, while the strength of Insulin comes from the data problem described above.

### Possible improvements
Use the original raw dataset, try Random Forest or Gradient Boosting, handle the class imbalance (class weights or SMOTE), and adjust the decision threshold to favour recall.